# Parts 3–4 · 03 Returns (part 4)

Abnormal returns around every event in `data/surprise.parquet`. There's no WRDS connection, and no dependency on Mahaut's files.

- **Day 0** is the reaction session from the release clock: the wire time where one exists, I/B/E/S otherwise (see `clock_status`).
- **Expected return:** Fama-French three factors plus momentum. Betas come from trading days −280 to −31 and need at least 120 returns; the intercept is not used.
- **`car_0_1`:** abnormal return on days 0 and +1, the market's reaction.
- **Post-release, "not yet priced":** enter at the first CRSP price after the release (day 0's open, or its close for intraday releases), then hold to day +1 and to day +60.
- **`ret_prior_1m`:** days −22 to −2, a baseline control.
- **Market-adjusted versions** of each serve as robustness checks.
- **Delistings:** the delisting return is compounded with that day's return (CRSP's daily return excludes it), and the event is flagged. A 60-day window that runs past CRSP's last date is left missing.

**Needs:** `common.py`, `returns.py` in `code/`; `data/surprise.parquet` (from 02); `cache/` (from 01).

**Writes:** `data/returns.parquet`, `results/returns_info.csv`, `results/sue_deciles_dev.csv`.

The checks at the end use **development years only (2006–2016)**. The holdout stays untouched.

In [18]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, sys
from pathlib import Path

TEAM_DIR = Path("/content/drive/MyDrive/S08 NLP")
PROJECT_DIR = TEAM_DIR / "3_4_surprise_returns"
os.environ["PROJECT_DIR"] = str(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "code"))

need = [PROJECT_DIR / "code" / f for f in ["common.py", "returns.py", "surprise.py"]]
need += [PROJECT_DIR / "data" / "surprise.parquet", PROJECT_DIR / "data" / "trading_calendar.csv"]
missing = [str(p) for p in need if not p.exists()]
assert not missing, f"missing: {missing}"
print("project folder:", PROJECT_DIR)

Mounted at /content/drive
project folder: /content/drive/MyDrive/S08 NLP/3_4_surprise_returns


## Load

In [19]:
import importlib, time
import numpy as np
import pandas as pd
import common, surprise as S, returns as Rm
for m in (common, S, Rm):
    importlib.reload(m)

t0 = time.time()
cal = common.load_trading_calendar(PROJECT_DIR / "data" / "trading_calendar.csv")
s = pd.read_parquet(PROJECT_DIR / "data" / "surprise.parquet")
px = S.load_cache("crsp_daily", columns=["permno", "date", "dlyret", "dlyprc", "dlyopen", "dlycumfacpr"])
px = px[px["permno"].isin(s["permno"].unique())]
ff = S.load_cache("ff_daily")
delists = S.load_cache("crsp_delists")
print(f"events {len(s):,} | CRSP rows {len(px):,} | factor days {len(ff):,} | "
      f"delistings {len(delists):,} | {time.time() - t0:.0f}s")

events 217,611 | CRSP rows 19,568,024 | factor days 5,722 | delistings 10,863 | 5s


## Does CRSP's daily return already include delisting returns?

The first run (4 Nov) showed it does not: only 207 of the 4,306 delisting dates with a daily return carry the delisting return. So the code compounds the two on the delisting date. Where they already match, it leaves the day alone, so nothing is counted twice.

In [20]:
for k, v in Rm.delisting_check(px, delists).items():
    print(f"  {k:<48} {v:,}")

  delistings with a delisting return               10,486
  with a daily return on the delisting date        4,306
  where that return equals the delisting return    207


## Compute

In [21]:
t0 = time.time()
ret, info = Rm.compute_returns(s, px, ff, cal, delists)
print(f"{time.time() - t0:.0f}s")
for k, v in info.items():
    print(f"  {k:<40} {v:,}" if isinstance(v, int) else f"  {k:<40} {v}")

cov = ret[["car_0_1_ff4", "car_0_1_mkt", "post_car_1_ff4", "post_car_60_ff4",
           "post_ret_60_raw", "ret_prior_1m"]].notna().mean().round(3)
print("\nshare of events with each measure:", cov.to_dict())
print("market beta:", ret["b_mkt"].describe(percentiles=[.1, .5, .9]).round(2).to_dict())
print("drift past CRSP's end (left missing):", int((~ret["drift_complete"]).sum()),
      "| delisted within 60 days:", int(ret["delisted_in_window"].sum()))

35s
  trading days in panel                    5,283
  stocks in panel                          7,763
  factors                                  factor units: decimal (median |MKTRF| 0.0052)
  delisting returns available              4,323
  delisting returns used alone (no daily return) 17
  delisting returns compounded with the day's return 4,099
  delisting returns already in the day's return 207
  events                                   217,611
  events with a singular estimation window 0
  events with betas                        212,310

share of events with each measure: {'car_0_1_ff4': 0.975, 'car_0_1_mkt': 1.0, 'post_car_1_ff4': 0.975, 'post_car_60_ff4': 0.963, 'post_ret_60_raw': 0.987, 'ret_prior_1m': 1.0}
market beta: {'count': 212310.0, 'mean': 0.98, 'std': 0.45, 'min': -7.34, '10%': 0.45, '50%': 0.98, '90%': 1.49, 'max': 17.52}
drift past CRSP's end (left missing): 2668 | delisted within 60 days: 2180


## Sanity checks: development years only

On the preview sample (3+ analysts, price above $5):
- Abnormal returns should average close to zero.
- The reaction should rise steadily from the most negative SUE decile to the most positive.
- The SUE-only rank IC is the baseline number the text signal has to beat.

The I/B/E/S stand-in clock is used throughout, so treat these numbers as provisional.

In [22]:
d = s.merge(ret.drop(columns=["permno", "reaction_date", "session_type", "entry_field"]), on="event_id")
d = d[d["n_analysts"].ge(3) & d["price_t2"].gt(5)]
d = common.guard_holdout(d)
d = d[d["split"].eq("dev")]
print(f"development-period events: {len(d):,}")

print("\nmean / median car_0_1 (FF4):", round(d["car_0_1_ff4"].mean(), 4), "/", round(d["car_0_1_ff4"].median(), 4))
dec = Rm.decile_means(d, "sue_price", ["car_0_1_ff4", "post_car_1_ff4", "post_car_60_ff4"])
print("\nby SUE decile (1 = most negative):")
print(dec.round(4).to_string())

print("\nSUE-only rank IC, averaged over announcement seasons (t_nw: Newey-West, 2 lags):")
for outcome in ["car_0_1_ff4", "post_car_1_ff4", "post_car_60_ff4"]:
    print(f"  {outcome:<18}", Rm.rank_ic_by_season(d, "sue_price", outcome))

# Placebos: returns BEFORE the reaction session. A large IC on day -1 would mean
# the clock is often a session late (the real reaction falling on our day -1).
# A small positive IC is normal: some news leaks or is anticipated.
print("\nplacebo, returns before the release (should be close to zero):")
for outcome in ["ar_m1", "ret_prior_1m"]:
    print(f"  {outcome:<18}", Rm.rank_ic_by_season(d, "sue_price", outcome))

# The two-day reaction absorbs a release placed one session off, so it can't
# test the clock. The post-release return can: if a "pre-open" release really
# came after the open, the open price is before the news and the "post-release"
# return contains the reaction itself. After-close releases are entered at the
# next morning's open, hours after the news, which makes them the clean benchmark.
print("\nIC by session type: reaction, and from the first price after the release to day +1")
for st, g in d.groupby("session_type"):
    if len(g) > 500:
        r = Rm.rank_ic_by_season(g, "sue_price", "car_0_1_ff4")
        p = Rm.rank_ic_by_season(g, "sue_price", "post_car_1_ff4")
        print(f"  {st:<12} n={len(g):>6,}  reaction IC {r['mean_ic']:.3f}   post-release IC {p['mean_ic']:.3f} (t {p['t_stat']})")

holdout guard: dropped 32,440 rows from 2021-2025
development-period events: 73,585

mean / median car_0_1 (FF4): 0.0009 / 0.0009

by SUE decile (1 = most negative):
        events  car_0_1_ff4  post_car_1_ff4  post_car_60_ff4
decile                                                      
1         7377      -0.0486         -0.0135          -0.0161
2         7357      -0.0347         -0.0068          -0.0073
3         7353      -0.0225         -0.0043          -0.0055
4         7353      -0.0110         -0.0027          -0.0034
5         7361      -0.0004          0.0003          -0.0004
6         7352       0.0078          0.0022           0.0028
7         7348       0.0160          0.0035           0.0058
8         7358       0.0241          0.0061           0.0045
9         7352       0.0332          0.0071           0.0078
10        7374       0.0457          0.0114           0.0155

SUE-only rank IC, averaged over announcement seasons (t_nw: Newey-West, 2 lags):
  car_0_1_ff4       

## Timing check by clock status

Since 02 now uses the wire time wherever one exists, each event carries `clock_status`:

- **wire_ibes_agree:** wire and I/B/E/S put the release in the same session (the two-source primary sample).
- **wire_ibes_conflict:** they disagree, and the wire is used.
- **ibes_only:** no wire time, so I/B/E/S stands in.

The first run (4 Nov) showed the day −1 IC is the same where the wire confirms the timing (0.052), so it's real pre-release movement, not a clock error. Conflicts were 1.6% of wire-covered events, and before the fix their "post-release" IC was 0.46: the reaction itself. With the wire time it should now look like the rest.

In [23]:
if "clock_status" not in d.columns:
    print("surprise.parquet has no clock_status: rerun 02_surprise first")
else:
    print("development-period events by clock status:", d["clock_status"].value_counts().to_dict(), "\n")
    for k, g in d.groupby("clock_status"):
        m1 = Rm.rank_ic_by_season(g, "sue_price", "ar_m1")
        r = Rm.rank_ic_by_season(g, "sue_price", "car_0_1_ff4")
        p = Rm.rank_ic_by_season(g, "sue_price", "post_car_1_ff4")
        q = Rm.rank_ic_by_season(g, "sue_price", "post_car_60_ff4")
        print(f"  {k:<19} n={len(g):>6,}  day -1 {m1['mean_ic']:.3f}  reaction {r['mean_ic']:.3f}  "
              f"post +1 {p['mean_ic']:.3f} (t_nw {p['t_nw']})  post +60 {q['mean_ic']:.3f} (t_nw {q['t_nw']})")

development-period events by clock status: {'ibes_only': 36972, 'wire_ibes_agree': 36041, 'wire_ibes_conflict': 572} 

  ibes_only           n=36,972  day -1 0.057  reaction 0.383  post +1 0.140 (t_nw 17.18)  post +60 0.066 (t_nw 8.55)
  wire_ibes_agree     n=36,041  day -1 0.051  reaction 0.351  post +1 0.086 (t_nw 10.68)  post +60 0.024 (t_nw 2.64)
  wire_ibes_conflict  n=   572  day -1 0.012  reaction 0.361  post +1 0.096 (t_nw 2.07)  post +60 0.132 (t_nw 3.26)


## Save

In [24]:
(PROJECT_DIR / "results").mkdir(exist_ok=True)
ret.to_parquet(PROJECT_DIR / "data" / "returns.parquet", index=False)
pd.Series(info).to_csv(PROJECT_DIR / "results" / "returns_info.csv", header=["value"])
dec.to_csv(PROJECT_DIR / "results" / "sue_deciles_dev.csv")
print(f"saved returns for {len(ret):,} events")

saved returns for 217,611 events
